In [58]:
import pandas as pd
import json
import random
import minsearch

from tqdm.auto import tqdm
from google import genai
from dotenv import load_dotenv
import os

In [59]:
load_dotenv()

client = genai.Client(
    api_key=os.getenv("GEMINI_API_KEY")
)

In [60]:
df = pd.read_csv("../data/processed/recipes_documents.csv")

df["id"] = range(len(df))

df.head()

,name,description,ingredients,steps,tags,minutes,document,id
0,arriba baked winter squash mexican style,autumn is my favorite time of year to cook! th...,"['winter squash', 'mexican seasoning', 'mixed ...","['make a choice and proceed with recipe', 'dep...","['60-minutes-or-less', 'time-to-make', 'course...",55,\nRecipe Name: arriba baked winter squash me...,0
1,a bit different breakfast pizza,this recipe calls for the crust to be prebaked...,"['prepared pizza crust', 'sausage patty', 'egg...","['preheat oven to 425 degrees f', 'press dough...","['30-minutes-or-less', 'time-to-make', 'course...",30,\nRecipe Name: a bit different breakfast pizz...,1
2,all in the kitchen chili,this modified version of 'mom's' chili was a h...,"['ground beef', 'yellow onions', 'diced tomato...","['brown ground beef in large pot', 'add choppe...","['time-to-make', 'course', 'preparation', 'mai...",130,\nRecipe Name: all in the kitchen chili\n\nDe...,2
3,alouette potatoes,"this is a super easy, great tasting, make ahea...","['spreadable cheese with garlic and herbs', 'n...",['place potatoes in a large pot of lightly sal...,"['60-minutes-or-less', 'time-to-make', 'course...",45,\nRecipe Name: alouette potatoes\n\nDescripti...,3
4,amish tomato ketchup for canning,my dh's amish mother raised him on this recipe...,"['tomato juice', 'apple cider vinegar', 'sugar...",['mix all ingredients& boil for 2 1 / 2 hours ...,"['weeknight', 'time-to-make', 'course', 'main-...",190,\nRecipe Name: amish tomato ketchup for cann...,4


In [61]:
df_eval = df.sample(
    n=10,
    random_state=42
).reset_index(drop=True)

df_eval.shape

(10, 8)

In [62]:
text_fields = [
    "document"
]

keyword_fields = [
    "name"
]

In [63]:
index = minsearch.Index(
    text_fields=text_fields,
    keyword_fields=keyword_fields
)

In [64]:
index.fit(df.to_dict(orient="records"))

In [65]:
results = index.search(
    query="vegetarian pasta",
    num_results=5
)

results

[{'name': 'easy pasta and vegetables',
  'description': 'yummy and easy pasta dish.  leftovers are great, but it does not freeze well.  whole family enjoys the dish.',
  'ingredients': "['pasta sauce', 'water', 'pasta', 'frozen vegetables', 'mozzarella cheese']",
  'steps': "['combine pasta sauce and water in a large skillet', 'bring to a boil', 'add pasta and vegetables', 'cover , reduce heat , and simmer 15 minutes or until pasta is tender', 'remove from heat and sprinkle with cheese']",
  'tags': "['30-minutes-or-less', 'time-to-make', 'course', 'main-ingredient', 'preparation', 'healthy', '5-ingredients-or-less', 'main-dish', 'pasta', 'easy', 'beginner-cook', 'low-fat', 'vegetarian', 'dietary', 'one-dish-meal', 'low-cholesterol', 'low-calorie', 'healthy-2', 'low-in-something', 'pasta-rice-and-grains']",
  'minutes': 20,
  'document': '\nRecipe Name: easy pasta and vegetables\n\nDescription:\nyummy and easy pasta dish.  leftovers are great, but it does not freeze well.  whole family

In [66]:
prompt1_template = """
You are a cooking expert generating evaluation questions.

For the recipe below, generate exactly 2 realistic questions that a user might ask.

Return ONLY a JSON array.

Each object must contain:
- id
- question

Recipe ID: {id}

Recipe Name:
{name}

Description:
{description}

Ingredients:
{ingredients}

Steps:
{steps}

Cooking Time:
{minutes} minutes
""".strip()

In [67]:
def llm_json(prompt):

    response = client.models.generate_content(
       model="gemini-3.1-flash-lite",
        contents=prompt
    )

    return json.loads(response.text)

In [68]:
row = df_eval.iloc[0]

prompt = prompt1_template.format(**row.to_dict())

print(prompt)

You are a cooking expert generating evaluation questions.

For the recipe below, generate exactly 2 realistic questions that a user might ask.

Return ONLY a JSON array.

Each object must contain:
- id
- question

Recipe ID: 59957

Recipe Name:
crab spinach casserole

Description:
another quick, easy, tasty casserole i have had for years

Ingredients:
['frozen chopped spinach', 'mayonnaise', 'flour', 'salt', 'fresh ground black pepper', 'garlic granules', 'sweet paprika', 'dried rosemary', 'dried thyme', 'onion', 'fresh mushrooms', 'milk', 'hard-boiled eggs', 'frozen crabmeat', 'swiss cheese']

Steps:
['spread spinach in bottom of a well buttered square 8" baking dish', 'combine mayonnaise , flour , salt , pepper , garlic , paprika , rosemary and thyme in a medium saucepan', 'add onion and mushrooms , and cook over medium heat until bubbly , stirring frequently', 'gradually add milk , and cook over medium heat until thickened , stirring constantly', 'stir in eggs', 'pour into baking di

In [69]:
questions = llm_json(prompt)

print(type(questions))
print(questions)

<class 'list'>
[{'id': 1, 'question': 'Do I need to thaw and drain the frozen spinach before spreading it in the baking dish?'}, {'id': 2, 'question': 'Can I substitute fresh crabmeat for the frozen crabmeat, or will it change the cooking time?'}]


In [70]:
results = []

for _, row in tqdm(df_eval.iterrows(), total=len(df_eval)):

    prompt = prompt1_template.format(**row.to_dict())

    questions = llm_json(prompt)

    for q in questions:
        q["id"] = row["id"]
        results.append(q)

  0%|          | 0/10 [00:00<?, ?it/s]

In [71]:
df_question = pd.DataFrame(results)

In [72]:
df_question.to_csv(
    "../data/processed/ground-truth-retrieval.csv",
    index=False
)

In [73]:
df_question = pd.read_csv(
    "../data/processed/ground-truth-retrieval.csv"
)

ground_truth = df_question.to_dict(orient="records")

In [74]:
def hit_rate(relevance_total):
    cnt = 0

    for line in relevance_total:
        if True in line:
            cnt += 1

    return cnt / len(relevance_total)

In [75]:
def mrr(relevance_total):
    total_score = 0.0

    for line in relevance_total:
        for rank in range(len(line)):
            if line[rank]:
                total_score += 1 / (rank + 1)
                break

    return total_score / len(relevance_total)

In [76]:
def search(query):
    results = index.search(
        query=query,
        num_results=10
    )

    return results

In [77]:
def evaluate(ground_truth, search_function):

    relevance_total = []

    for q in tqdm(ground_truth):

        doc_id = q["id"]

        results = search_function(q["question"])

        relevance = [d["id"] == doc_id for d in results]

        relevance_total.append(relevance)

    return {
        "hit_rate": hit_rate(relevance_total),
        "mrr": mrr(relevance_total)
    }

In [78]:
from tqdm.auto import tqdm

def evaluate(ground_truth, search_function):

    relevance_total = []

    for q in tqdm(ground_truth):

        doc_id = q["id"]

        results = search_function(q["question"])

        relevance = [d["id"] == doc_id for d in results]

        relevance_total.append(relevance)

    return {
        "hit_rate": hit_rate(relevance_total),
        "mrr": mrr(relevance_total)
    }

In [79]:
evaluate(
    ground_truth,
    search
)

  0%|          | 0/20 [00:00<?, ?it/s]

{'hit_rate': 0.2, 'mrr': 0.04464285714285714}